# VC4 · Demo en directo — Smart City
## "El pipeline salió en VERDE. Y los datos son basura. ¿Cómo?"

**Esto NO es la actividad evaluable.** Es la demostración de la videoconferencia del NF4.
Tu actividad instrumenta **el servidor de un juego**; aquí monitorizamos una red de sensores.
Mismo trabajo, otros datos. **No se entrega.**

**Antes de ejecutar nada**, una sola vez, desde la raíz del repositorio:

```bash
python demo/nf4_smartcity/preparar_demo.py
```

Y **no hace falta Docker ni Grafana.** Todo el stack que importa —instrumentar, exponer
`/metrics`, evaluar el SLO, escribir las alertas como código— corre aquí, en Python.

---

### El detonante

El pipeline de sensores de anoche terminó **en verde**. `SUCCESS`. Su marcador `_SUCCESS`
escrito, cero excepciones, memoria holgada. Por todos los controles que aprendiste en el NF3,
**está perfecto**.

Y esta mañana, el panel de la ciudad muestra datos que no tienen sentido. Alguien pregunta lo
de siempre: *"¿pero no había salido bien?"*. Sí. Salió bien. Y está roto.

Vamos a entender cómo pueden ser verdad las dos cosas a la vez — y a construir lo que lo habría
detectado.

In [ ]:
import os, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

try:
    BASE = Path(__file__).parent
except NameError:
    BASE = Path.cwd()
    if BASE.name != "nf4_smartcity":
        BASE = BASE / "demo" / "nf4_smartcity"
RAW = BASE / "raw"

assert (RAW / "lote_hoy.csv").exists(), (
    "No encuentro los datos. Ejecuta una vez, desde la raíz del repo:\n"
    "    python demo/nf4_smartcity/preparar_demo.py"
)
print("Datos listos en:", BASE)

---
# ACTO 0 · El pipeline verde y roto

Empecemos por el final: el lote de hoy, el que "salió bien".

In [ ]:
lote = pd.read_csv(RAW / "lote_hoy.csv")
manifiesto = json.load(open(RAW / "manifiesto.json"))

# Los controles del NF3: ¿hay nulos? ¿valores fuera de rango? ¿tipos raros?
print("nulos            :", int(lote.isna().sum().sum()))
print("lecturas < 0     :", int((lote['lectura'] < 0).sum()))
print("lecturas > 100   :", int((lote['lectura'] > 100).sum()))
print("¿pasa el NF3?    : SÍ. Cada fila que hay es impecable.")

**Todos los tests del NF3 pasan.** No hay nulos, ni rangos rotos, ni tipos raros. Lo que llegó
es válido. Y sin embargo:

In [ ]:
esperados = manifiesto["sensores_esperados"]
hoy = manifiesto["sensores_hoy"]

print(f"sensores esperados (día normal): {esperados}")
print(f"sensores que reportaron hoy    : {hoy}")
print(f"                                  ^^^ faltan {esperados - hoy} sensores ENTEROS")
print()
print(f"filas hoy         : {manifiesto['filas_hoy']:,}")
print(f"filas un día normal: {manifiesto['filas_dia_normal']:,}")

Ahí está el crimen perfecto. **Tres sensores llevan horas mudos** y el pipeline no se ha
inmutado, porque procesó **lo que le llegó** — y lo que le llegó, de 47 sensores, era perfecto.

> Aquí está la frontera entera entre los dos núcleos, en un dato:
>
> **NF3 pregunta: "¿está bien lo que llegó?"** → Sí. **NF4 pregunta: "¿llegó todo?"** → No.
>
> Un test de rango mira cada fila y dice "válida". **Nunca** te dirá que faltan tres sensores,
> porque las filas que hay son correctas. Eso solo lo caza un **monitor**: algo que compara el
> **volumen de hoy contra el histórico** y grita cuando 1.128 debería haber sido 1.200.

Un test pregunta *"¿está bien ahora que lo miro?"*. Un monitor pregunta *"¿está como siempre?"*.
El resto de la sesión es construir monitores.

---
# ACTO 1 · Instrumentar: enseñarle al pipeline a hablar de sí mismo

Un monitor necesita métricas. Y las métricas no se sacan del dato: **se sacan del pipeline
mientras trabaja**. Vamos a instrumentarlo con `prometheus_client` —la librería real de
producción— y a mirar lo único que Prometheus llega a ver.

In [ ]:
import sys
sys.path.insert(0, str(BASE))
from pipeline_demo import procesar, metricas_texto, latencias_demo

# Procesamos en modo normal y miramos qué EXPONE el pipeline
reg = procesar("normal")
print(metricas_texto(reg)[:900])

In [ ]:
# Hacemos lo que haría Prometheus+Grafana: leer /metrics y pintarlo como un panel.
from prometheus_client.parser import text_string_to_metric_families
from IPython.display import HTML, display

def leer_metricas(reg):
    v = {}
    for fam in text_string_to_metric_families(metricas_texto(reg)):
        for s in fam.samples:
            v[s.name] = s.value
    return v

SLO = {"error_rate_max": 0.05, "frescura_max_s": 60, "latencia_p95_max_s": 0.5}

def tarjeta(titulo, valor, ok, slo_txt):
    color = "#1a7f37" if ok else "#cf222e"
    fondo = "#f2fdf5" if ok else "#fff5f5"
    luz   = "OK" if ok else "SLO ROTO"
    return f"""
    <div style="border:1px solid #e0e0e0;border-radius:12px;padding:16px 20px;
                min-width:170px;background:{fondo};font-family:system-ui">
      <div style="font-size:12px;color:#666;text-transform:uppercase;letter-spacing:.5px">{titulo}</div>
      <div style="font-size:30px;font-weight:700;color:{color};margin:4px 0">{valor}</div>
      <div style="font-size:11px;color:#888">{slo_txt} · <b style="color:{color}">{luz}</b></div>
    </div>"""

def torre_control(modo):
    v = leer_metricas(procesar(modo))
    er = v["errores_total"] / v["eventos_procesados_total"]
    fr = v["datos_frescura_segundos"]
    cartas = [
        tarjeta("Tasa de error", f"{er*100:.1f}%", er <= SLO["error_rate_max"], "SLO &le; 5%"),
        tarjeta("Frescura del dato", f"{fr:.0f}s", fr <= SLO["frescura_max_s"], "SLO &le; 60s"),
        tarjeta("Eventos procesados", f"{v['eventos_procesados_total']:.0f}", True, "throughput"),
    ]
    display(HTML(f"""
      <div style="font-family:system-ui;margin:4px 0 6px;font-weight:600">
        Torre de control &middot; modo <span style="text-transform:uppercase">{modo}</span></div>
      <div style="display:flex;gap:12px;flex-wrap:wrap">{''.join(cartas)}</div>"""))

torre_control("normal")

Eso es un endpoint `/metrics`: texto plano que el pipeline publica sobre sí mismo. Fíjate en los
**tres tipos** que aparecen, porque son los que existen:

- **Counter** (`eventos_procesados_total`, `errores_total`): solo sube. Cuentas acumuladas.
- **Gauge** (`datos_frescura_segundos`, `cola_pendiente`): sube y baja. Una foto del ahora.
- **Histogram** (`latencia_..._bucket`): reparte los valores en cajones. Sirve para percentiles
  — y ahora verás por qué eso importa.

> **En el directo:** aquí se lanza `python pipeline_demo.py --modo normal --puerto 8000` en una
> terminal y se abre `/metrics` en el navegador (pestaña *Ports* de Codespaces). Es la Spark UI
> del NF4: ver el endpoint vivo. En el cuaderno lo leemos como texto para que quede grabado.

---
# ACTO 2 · La media miente (otra vez). Por eso el SLO es p95

Nuestro SLO de latencia es **p95 ≤ 0,5 s**. ¿Por qué p95 y no "latencia media ≤ 0,5 s"? Diez
peticiones de ejemplo:

In [ ]:
lat = np.array(latencias_demo())
print("latencias (s):", [round(float(x), 2) for x in lat])
print()
print(f"media : {lat.mean():.3f} s   -> SLO 0.5s: {'PASA ✅' if lat.mean()<=0.5 else 'FALLA ❌'}")
print(f"p95   : {np.percentile(lat, 95):.3f} s   -> SLO 0.5s: {'PASA ✅' if np.percentile(lat,95)<=0.5 else 'FALLA ❌'}")

Uno de cada diez usuarios ha esperado **cuatro segundos**, y la media dice que todo va bien.

Es **el punto de ruptura del NF3**, otra vez, ahora en latencia: la media **suma**, así que
diluye. Y la dilución aquí tiene una consecuencia concreta y cara:

> **La latencia media esconde justo a los usuarios que se están yendo.** Nadie abandona por la
> media. Abandonan los de la cola. Por eso la fiabilidad se mide en **percentiles**: p50 (la
> experiencia típica), **p95 (el SLO estándar)**, p99 (servicios críticos).

Simetría con el NF3, y es el arco del módulo: allí usabas la **mediana** para fijar la
normalidad sin dejarte engañar por picos; aquí usas el **p95** para medir la cola que la media
esconde. La misma desconfianza hacia el promedio, aplicada a las dos mitades.

---
# ACTO 3 · El incidente: la alerta se decide sola

Ahora activamos el modo incidente. En producción no hay un interruptor —hay una degradación a
la que llegas tarde—, pero para *verlo* lo provocamos. Los tres SLO a la vez.

In [ ]:
# leer_metricas() y SLO ya están definidos (ACTO 1). Aquí solo comparamos y pintamos.
print(f"{'modo':10} {'error_rate':>11} {'frescura':>9} {'¿alerta?':>9}")
for modo in ["normal", "incidente"]:
    v = leer_metricas(procesar(modo))
    error_rate = round(v["errores_total"] / v["eventos_procesados_total"], 4)
    frescura = v["datos_frescura_segundos"]
    alerta = error_rate > SLO["error_rate_max"] or frescura > SLO["frescura_max_s"]
    print(f"{modo:10} {error_rate:>11} {frescura:>8.0f}s {('SÍ 🔴' if alerta else 'no  '):>9}")

# Y el mismo panel del ACTO 1, ahora en incidente: mira cómo se pone en rojo.
torre_control("incidente")

La decisión **no la toma un humano mirando un gráfico**: la toma una condición sobre la métrica.
`error_rate 0,33 > 0,05` → alerta. Eso es un SLO **ejecutándose**. Y ahora lo escribimos como
código, para que corra solo, para siempre, versionado en git:

In [ ]:
print((BASE / "observabilidad" / "alertas.yml").read_text()[:1100])

Tres piezas en cada regla, y cada una es una decisión de ingeniería:

- **`expr`** — el SLI comparado con el SLO. **La regla ES el SLO, escrito.**
- **`for`** — el antídoto contra el ruido. `for: 1m` dice *"la condición debe sostenerse un
  minuto"*. Sin él, un pico de dos segundos te despierta (flapping). Más `for` = menos ruido
  pero te enteras más tarde: por eso `DatoObsoleto` lleva `for: 2m` (un dato viejo no es una
  emergencia de segundos) y `TasaErroresAlta` lleva `for: 1m`.
- **`severity`** — la política, escrita. `critical` te despierta a las 3:00; `warning` espera a
  mañana. **Es la misma idea que el `severity: error`/`warn` de dbt en el NF3.** Decidirlo
  *antes* del incidente es lo que evita improvisar medio dormido.

Validamos que el YAML es correcto (en un CI real: `promtool check rules`; aquí, Python):

In [ ]:
import yaml
doc = yaml.safe_load((BASE / "observabilidad" / "alertas.yml").read_text())
reglas = doc["groups"][0]["rules"]
print(f"YAML válido · {len(reglas)} reglas:\n")
for r in reglas:
    ok = all(k in r for k in ("alert", "expr", "labels", "annotations"))
    print(f"  {'✅' if ok else '❌'} {r['alert']:18} severity={r['labels']['severity']:9} for={r.get('for','—')}")

---
# ACTO 4 · La alerta que nadie escribe: el silencio

Y ahora la pregunta del núcleo, la que casi todo el mundo olvida. Todas las reglas de arriba
tienen la misma forma: *"si la métrica cruza el umbral, avisa"*.

**¿Y si no hay métrica?**

No el incidente. La **muerte**. El pipeline no se degrada: se cae. Deja de exponer `/metrics`.
Veamos qué pasa entonces con nuestras alertas de umbral.

In [ ]:
# El pipeline ha muerto: no hay registry, no hay métricas. Simulamos ese vacío.
metricas_de_un_muerto = {}    # /metrics no responde

def evaluar_umbrales(v):
    disparos = []
    if v:
        er = v.get("errores_total", 0) / max(v.get("eventos_procesados_total", 1), 1)
        if er > SLO["error_rate_max"]:                 disparos.append("TasaErroresAlta")
        if v.get("datos_frescura_segundos", 0) > SLO["frescura_max_s"]: disparos.append("DatoObsoleto")
    return disparos

print("Pipeline MUERTO. ¿Qué alertas de umbral saltan?")
print("  ->", evaluar_umbrales(metricas_de_un_muerto) or "NINGUNA")

**Ninguna.** Léelo despacio, porque es lo más importante del núcleo:

> Un umbral no puede saltar si **no llega ningún valor que lo cruce**. Si el pipeline se muere,
> no hay tasa de error alta: **no hay tasa de error**. El dashboard, en vez de ponerse rojo, se
> queda **en blanco**. Todas tus alertas, calladas.
>
> **Un sistema muerto es un sistema silencioso. Y el silencio se parece muchísimo a que todo va
> bien.**

Por eso hacen falta alertas sobre **la ausencia**. Son dos, y ya estaban en nuestro
`alertas.yml` (la regla `PipelineCaido`):

In [ ]:
# up == 0            -> el scrape falla (Prometheus fue a buscar y no encontró a nadie)
# absent(metrica)    -> la serie ha desaparecido
regla_silencio = [r for r in reglas if r["alert"] == "PipelineCaido"][0]
print("La alerta del silencio:")
print(f"  expr: {regla_silencio['expr']}")
print(f"  -> {regla_silencio['annotations']['description']}")

Y fíjate en la elegancia: `up` **no la genera tu código**. La genera **el hecho de que
Prometheus vaya a buscar** (el modelo *pull* del §4.3). Prometheus sabe cuándo no encuentra a
nadie. `up == 0` es la métrica más importante que tienes y es gratis.

Su versión en el mundo de los datos es **justo el ACTO 0**: si esperas 50 sensores y hoy
reportan 47, **hay tres callados** y ninguna alerta de umbral te lo dirá, porque los 47 que
quedan van estupendamente. **La media de los que hablan nunca te avisa de los que se callaron.**
Contar cuántas entidades reportan es la alerta del silencio, aplicada a tus datos.

---
# ACTO 5 · Todo esto, como código (y por qué Grafana no lleva capturas)

Podríamos haber construido el dashboard clicando en Grafana. Funciona. Y es exactamente lo que
**no** hemos hecho. Nuestro panel es un fichero:

In [ ]:
dash = json.load(open(BASE / "observabilidad" / "torre_control.json"))
print("torre_control.json ·", len(dash["panels"]), "paneles (como código):\n")
for p in dash["panels"]:
    print(f"  [{p['id']}] {p['title']:32} {p['targets'][0]['expr'][:55]}")

| Clicando en Grafana | **Como código** (`alertas.yml`, `torre_control.json`) |
|---|---|
| Nadie sabe quién lo cambió | **Está en git**: historial, autor, motivo |
| No se puede revisar | Se revisa en un *pull request* |
| No se replica en otro entorno | Se despliega igual en dev y en prod |
| Se pierde si alguien borra el panel | Se restaura del repositorio |
| No se puede probar | `promtool check rules` en el CI |

> Por eso tu entrega **no lleva capturas de pantalla**. Una captura demuestra que un día viste
> un gráfico. `alertas.yml` y `torre_control.json` demuestran que **el sistema lo hace solo,
> siempre, y que otro equipo puede desplegarlo mañana**. Esa es la diferencia entre *haber usado
> Grafana* y *haber hecho ingeniería de fiabilidad*.

Y esto **ya lo hiciste** en el NF3 con `schema.yml`: declarar en YAML, versionar en git,
ejecutar solo. Allí era calidad; aquí es vigilancia. La misma idea, la otra mitad del problema.

*(Si quieres ver el dashboard renderizado en Grafana, hay una captura en
`observabilidad/README.md`. Pero el entregable es el JSON, no la imagen.)*

---
## Lo que ha pasado en esta hora

Un pipeline verde y roto. Y la maquinaria para no volver a fiarte del verde:

| Acto | La idea | Teoría |
|---|---|---|
| 0 | NF3 tests vs NF4 monitores: "¿llegó todo?" | §4.0, §4.6 |
| 1 | Instrumentar: Counter, Gauge, Histogram | §4.2 |
| 2 | La media miente: SLO en p95, no en media | §4.2 |
| 3 | El SLO como código: `expr`, `for`, `severity` | §4.4, §4.5 |
| 4 | **La alerta del silencio**: `up`, `absent()` | §4.5 |
| 5 | Observabilidad como código, sin capturas | §4.6 |

La idea que las une: **la observabilidad clásica vigila el sistema; la de datos vigila el
dato.** Frescura, volumen, esquema, distribución. Y todo como código.

**Y ahora te toca a ti**, instrumentando el servidor de un juego. Mismo trabajo, otros datos:
tus tres SLO, tu `alertas.yml`, tu `torre_control.json`.

> **El gancho del NF5:** ya tienes datos que guardas bien (NF1), procesas a escala (NF2), sabes
> que son fiables (NF3) y vigilas en el tiempo (NF4). Toda la ingeniería está hecha. Y no sirve
> de **nada** si nadie decide con ella. El último núcleo es el más difícil: convertir todo esto
> en algo que un humano mire y **actúe**.